# 01 — Data Collection
Pull AQI data from EPA AirNow and weather data from NOAA CDO, then save to `data/raw/`.

**Before running:** Set your API keys in a `.env` file at the project root:
```
AIRNOW_API_KEY=your-key
NOAA_API_KEY=your-token
```

In [2]:
import sys
import pandas as pd
sys.path.append('..')

import os
from datetime import date
from dotenv import load_dotenv

load_dotenv('../.env')

from utils.airnow import fetch_date_range
from utils.noaa import find_station, fetch_weather

os.makedirs('../data/raw', exist_ok=True)

## Config — Change These

In [5]:
ZIP_CODE  = '98005'       # Change to your city's ZIP code
START     = date(2024, 1, 1)
END       = date(2024, 12, 31)
POLLUTANT = 'PM2.5'       # Options: 'PM2.5' or 'OZONE'

## 1. Pull AirNow AQI Data
This fetches one day at a time (AirNow's historical endpoint is per-day).
3 years of data takes ~5–10 minutes. Run once, then load from CSV.

In [6]:
aqi_df = fetch_date_range(ZIP_CODE, START, END, pollutant=POLLUTANT, sleep_sec=2.0)
aqi_df.head(10)

Fetching 366 days of AirNow data for ZIP 98005...
Done. 366 records collected.


,date,aqi,pollutant,category
0,2024-01-01,71,PM2.5,Moderate
1,2024-01-02,65,PM2.5,Moderate
2,2024-01-03,51,PM2.5,Moderate
3,2024-01-04,36,PM2.5,Good
4,2024-01-05,53,PM2.5,Moderate
5,2024-01-06,32,PM2.5,Good
6,2024-01-07,55,PM2.5,Moderate
7,2024-01-08,35,PM2.5,Good
8,2024-01-09,36,PM2.5,Good
9,2024-01-10,33,PM2.5,Good


In [ ]:
aqi_df.to_csv('../data/raw/aqi_raw.csv', index=False)
print(f'Saved {len(aqi_df)} rows to data/raw/aqi_raw.csv')

Saved 366 rows to data/raw/aqi_raw.csv


## 2. Pull NOAA Weather Data
Find the nearest station with good data coverage, then pull daily summaries.

In [6]:
station_id = find_station(ZIP_CODE, START, END)
print('Station ID:', station_id)

# If None, look one up manually at: https://www.ncei.noaa.gov/cdo-web/search
# Then hardcode it: station_id = 'GHCND:USW00024233'

Station ID: None


In [ ]:
weather_df = fetch_weather(station_id, START, END)
weather_df.head(10)

  Fetched weather 2024-01-01 → 2024-12-31: 1000 records


datatype,date,wind_ms,prcp_mm,tmax_c,tmin_c,tavg_c
0,2024-01-01,2.7,0.779685,26.559259,14.693564,20.626412


In [13]:
import requests
import pandas as pd
import os

# Seattle-Tacoma International Airport — reliable GHCND station
STATION_ID = "GHCND:USW00024233"

params = {
    "datasetid": "GHCND",
    "stationid": STATION_ID,
    "datatypeid": "TMAX,TMIN,PRCP,AWND",
    "startdate": "2024-01-01",
    "enddate": "2024-12-31",
    "limit": 1000,
    "units": "metric",
}
resp = requests.get(
    "https://www.ncei.noaa.gov/cdo-web/api/v2/data",
    params=params,
    headers={"token": os.getenv("NOAA_API_KEY")},
)
raw = resp.json()["results"]
df_long = pd.DataFrame(raw)
df_long = df_long[df_long['station'] == STATION_ID]
df_long["date"] = pd.to_datetime(df_long["date"]).dt.normalize()

weather_df = df_long.pivot_table(
    index="date", columns="datatype", values="value", aggfunc="first"
).reset_index()

weather_df.columns.name = None
weather_df = weather_df.rename(columns={
    "TMAX": "tmax_c", "TMIN": "tmin_c",
    "PRCP": "prcp_mm", "AWND": "wind_ms"
})
weather_df["tavg_c"] = (weather_df["tmax_c"] + weather_df["tmin_c"]) / 2

print(weather_df.shape)
weather_df.head()

(251, 6)


,date,wind_ms,prcp_mm,tmax_c,tmin_c,tavg_c
0,2024-01-01,1.2,0.0,8.3,1.1,4.70
1,2024-01-02,1.7,6.1,8.3,3.3,5.80
2,2024-01-03,3.3,2.5,10.0,5.0,7.50
3,2024-01-04,2.6,3.6,7.8,5.0,6.40
4,2024-01-05,6.1,5.8,8.3,4.4,6.35


In [14]:
print(station_id)

None


In [15]:
weather_df.to_csv('../data/raw/weather_raw.csv', index=False)
print(f'Saved {len(weather_df)} rows to data/raw/weather_raw.csv')

Saved 251 rows to data/raw/weather_raw.csv


In [16]:
aqi_df = pd.read_csv('../data/raw/aqi_raw.csv', parse_dates=['date'])
print(f'Loaded {len(aqi_df)} rows')
aqi_df.head()

Loaded 366 rows


,date,aqi,pollutant,category
0,2024-01-01,71,PM2.5,Moderate
1,2024-01-02,65,PM2.5,Moderate
2,2024-01-03,51,PM2.5,Moderate
3,2024-01-04,36,PM2.5,Good
4,2024-01-05,53,PM2.5,Moderate


## 3. Sanity Check

In [17]:
print('AQI shape:    ', aqi_df.shape)
print('Weather shape:', weather_df.shape)
print('\nAQI date range:    ', aqi_df['date'].min(), 'to', aqi_df['date'].max())
print('Weather date range:', weather_df['date'].min(), 'to', weather_df['date'].max())
print('\nMissing AQI values:\n',     aqi_df.isnull().sum())
print('\nMissing weather values:\n', weather_df.isnull().sum())

AQI shape:     (366, 4)
Weather shape: (251, 6)

AQI date range:     2024-01-01 00:00:00 to 2024-12-31 00:00:00
Weather date range: 2024-01-01 00:00:00 to 2024-09-07 00:00:00

Missing AQI values:
 date         0
aqi          0
pollutant    0
category     0
dtype: int64

Missing weather values:
 date       0
wind_ms    0
prcp_mm    2
tmax_c     0
tmin_c     2
tavg_c     2
dtype: int64
